# Демо-анализ: нагрузка, пропущенные, аномалии, операторы

Четыре графика на демо-данных SQL-части — той же базе, по которой
работает `analytics.py`. Данные сгенерированы, но с узнаваемой картиной
контакт-центра: суточный профиль, тихие выходные, проекты разного
размера и один заложенный сбой.

Перед запуском, из корня репозитория:

    python generate_demo.py
    python load_daily.py

Период демо заканчивается вчерашним днём, поэтому даты и цифры от
запуска к запуску меняются. Выводы под графиками не вписаны руками —
их печатает код по тем данным, что лежат в базе.

## Ячейка 1 — настройки

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from sqlalchemy import create_engine, inspect, text

from analytics import find_anomalies, hour_profile, load_daily
from config import DB_URL, MIN_CALLS_PER_DAY

# Проект для графиков нагрузки и операторов — самый крупный в демо
MAIN_PROJECT = "alpha"

# Долю пропущенных в часе считаем, только если за период в этот час
# пришло хотя бы столько звонков: «50%» от двух звонков — это один
# звонок, а не показатель
MIN_CALLS_FOR_SHARE = 30

# Час, с которого до которого работает дневная смена (включительно)
DAY_SHIFT = (8, 19)

# Цвета. Проект всегда одного цвета на всех графиках: цвет — это
# «кто», а не «какой по счёту». Поэтому на графиках одного проекта
# будни/выходные и смены различаются не цветом, а стилем линии и
# оттенком. Красный — только для сбоя, и рядом с ним всегда подпись:
# цвет в одиночку смысл не несёт.
PROJECT_COLORS = {"alpha": "#2a78d6", "beta": "#eb6834", "gamma": "#1baf7a"}
LIGHT_BLUE = "#86b6ef"               # тот же синий, светлее: ночная смена
ALERT = "#d03b3b"
INK, INK_2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"

# Общий вид всех графиков. rcParams — словарь настроек matplotlib по
# умолчанию: задаём один раз здесь, а не в каждом графике.
plt.rcParams.update({
    "figure.figsize": (9, 3.6),
    "figure.dpi": 110,
    "axes.spines.top": False,        # рамка сверху и справа не нужна
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.grid": True,
    "axes.grid.axis": "y",           # только горизонтальная сетка
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelcolor": INK_2,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "lines.linewidth": 2,
    "legend.frameon": False,
})

engine = create_engine(DB_URL)
if "calls" not in inspect(engine).get_table_names():
    raise RuntimeError("База пустая. Сначала: python generate_demo.py, "
                       "потом python load_daily.py")

# Дневная картина по проектам — та же функция, что у analytics.py:
# звонки, пропущенные, доля ответа, тип дня
daily = load_daily(engine)

# Проекты, по которым вообще есть смысл строить доли: мелкие отсеиваем
# тем же порогом, что и поиск аномалий
per_day = daily.groupby("project")["total"].mean()
BIG_PROJECTS = [p for p in PROJECT_COLORS if per_day.get(p, 0) >= MIN_CALLS_PER_DAY]

print(f"Период: {daily['call_date'].min():%d.%m.%Y} — "
      f"{daily['call_date'].max():%d.%m.%Y}, дней {daily['call_date'].nunique()}")
print("Звонков в день по данным телефонии, в среднем:",
      per_day.round(0).astype(int).to_dict())

## 1. Нагрузка по часам

Сколько звонков приходит в каждый час — отдельно для будних и
выходных. Их нельзя смешивать: у выходного свой режим, и средний час
по всем дням не описал бы ни один из них.

In [ ]:
h = pd.read_sql(text("SELECT call_date, call_hour, total FROM hourly "
                     "WHERE project = :p"), engine,
                params={"p": MAIN_PROJECT})
h["Тип дня"] = np.where(pd.to_datetime(h["call_date"]).dt.weekday >= 5,
                        "выходной", "будний")

# Среднее по дням одного типа: строки — часы, колонки — тип дня
profile = h.groupby(["call_hour", "Тип дня"])["total"].mean().unstack()

# fig — вся картинка, ax — область с осями, на которой рисуем
color = PROJECT_COLORS[MAIN_PROJECT]
fig, ax = plt.subplots()
for kind, style in [("будний", "-"), ("выходной", "--")]:
    ax.plot(profile.index, profile[kind], color=color, linestyle=style,
            label=kind)
    # Подпись прямо у пика линии: не надо искать легенду глазами.
    # xytext — сдвиг подписи от точки, в пунктах экрана
    peak = profile[kind].idxmax()
    ax.annotate(kind, (peak, profile.loc[peak, kind]), xytext=(8, 4),
                textcoords="offset points", color=INK_2)

ax.set_title(f"{MAIN_PROJECT}: сколько звонков приходит в каждый час")
ax.set_xticks(range(0, 24, 3))
ax.set_xticklabels([f"{x}:00" for x in range(0, 24, 3)])
ax.set_ylabel("звонков в час, в среднем")
ax.set_ylim(bottom=0)
ax.legend(loc="upper right")
plt.show()

peak = profile["будний"].idxmax()
share = profile["выходной"].sum() / profile["будний"].sum()
night = profile.loc[list(range(0, 7)), "будний"].sum() / profile["будний"].sum()
print(f"Пик в будни — {peak:02d}:00–{peak + 1:02d}:00, "
      f"в среднем {profile.loc[peak, 'будний']:.0f} звонков.")
print(f"Выходной день — примерно {share:.0%} от буднего по числу звонков.")
print(f"С полуночи до 7 утра приходит {night:.1%} дневного потока: "
      f"ночная смена нужна небольшая.")

## 2. Доля пропущенных по часам

Какая часть звонков в этот час осталась без ответа. По панели на
проект, шкала у всех общая — так проекты сравниваются честно.
Пунктир — доля пропущенных у проекта за сутки в целом, она же в
заголовке панели.
Часы, где звонков за период меньше `MIN_CALLS_FOR_SHARE`, не показаны.

In [ ]:
vh = pd.read_sql("SELECT project, call_hour, total, lost FROM v_hourly", engine)
vh["share"] = np.where(vh["total"] >= MIN_CALLS_FOR_SHARE,
                       vh["lost"] / vh["total"], np.nan)

# subplots(1, N) — N панелей в ряд; sharey=True — одна шкала на всех
fig, axes = plt.subplots(1, len(BIG_PROJECTS), figsize=(10, 3.4), sharey=True)
for ax, project in zip(np.atleast_1d(axes), BIG_PROJECTS):
    p = vh[vh["project"] == project].sort_values("call_hour")
    ax.bar(p["call_hour"], p["share"], width=0.8,
           color=PROJECT_COLORS[project])
    base = p["lost"].sum() / p["total"].sum()
    # axhline — горизонтальная линия через весь график. Её значение
    # пишем в заголовок панели: рядом с линией подпись задевала столбцы
    ax.axhline(base, color=INK_2, linestyle="--", linewidth=1)
    ax.set_title(f"{project}: за сутки {base:.0%}")
    ax.set_xticks([0, 6, 12, 18])
    ax.set_xticklabels(["0:00", "6:00", "12:00", "18:00"])
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))   # 0.25 -> 25%

fig.suptitle("Доля пропущенных звонков по часам", x=0.01, ha="left",
             fontweight="bold")
fig.tight_layout()
plt.show()

# Худшие часы — готовая функция из analytics.py, это же таблица к графику
display(hour_profile(engine))

for project in BIG_PROJECTS:
    p = vh[(vh["project"] == project) & vh["share"].notna()]
    base = p["lost"].sum() / p["total"].sum()
    # Тот же порог, что в sheets_analytics: час хуже обычного в 1,25 раза
    bad = p[p["share"] >= base * 1.25]["call_hour"].tolist()
    print(f"{project}: в среднем пропускается {base:.0%} звонков; "
          f"заметно хуже — в {', '.join(f'{x}:00' for x in bad) or 'нет таких часов'}")

## 3. Аномальные дни

Аномалии ищет `find_anomalies` из `analytics.py` — с его правилами:
отклонение не меньше трёх сигм по медиане и MAD, не меньше трети от
нормы, будни и выходные отдельно, подряд идущие дни — одно событие.
На графике — доля дозвонившихся по дням у проекта, где найдено
событие; сбой выделен красной полосой.

In [ ]:
events = find_anomalies(daily)
display(events if len(events) else "Аномалий не найдено")


def event_days(when: str) -> tuple:
    """
    «15.09 — 17.09.2026» или «15.09.2026» -> (первый день, последний).

    find_anomalies отдаёт период строкой для людей, а графику нужны
    даты. Год есть только у последней даты — первой его подставляем.
    """
    parts = when.split(" — ")
    last = pd.to_datetime(parts[-1], format="%d.%m.%Y")
    if len(parts) == 1:
        return last, last
    first = pd.to_datetime(f"{parts[0]}.{last.year}", format="%d.%m.%Y")
    if first > last:                  # событие на стыке декабря и января
        first = first - pd.DateOffset(years=1)
    return first, last


project = events["Проект"].iloc[0] if len(events) else "beta"
g = daily[daily["project"] == project].sort_values("call_date")
norm = g["answer_share"].median()

fig, ax = plt.subplots()
ax.plot(g["call_date"], g["answer_share"], color=PROJECT_COLORS[project],
        marker="o", markersize=6)
ax.axhline(norm, color=INK_2, linestyle="--", linewidth=1)
# Подпись справа за краем графика: внутри точки лежат как раз у нормы.
# xycoords: по X — доля ширины осей (1 = правый край), по Y — данные
ax.annotate(f"обычно {norm:.0%}", (1, norm),
            xycoords=("axes fraction", "data"), xytext=(4, 0),
            textcoords="offset points", va="center", color=INK_2)

for _, e in events[events["Проект"] == project].iterrows():
    first, last = event_days(e["Когда"])
    # axvspan — вертикальная полоса по оси X: от полудня накануне до
    # полудня после, чтобы точки сбоя оказались внутри неё
    ax.axvspan(first - pd.Timedelta(hours=12), last + pd.Timedelta(hours=12),
               color=ALERT, alpha=0.12, linewidth=0)
    # Подпись справа от полосы, на уровне провала: там пусто
    low = g.loc[g["call_date"].between(first, last), "answer_share"].min()
    ax.annotate(f"сбой: {e['Насколько сильно']}", (last, low),
                xytext=(14, 0), textcoords="offset points", va="center",
                color=ALERT)

ax.set_title(f"{project}: доля звонков, на которые ответили, по дням")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%d.%m"))
plt.show()

if len(events):
    e = events.iloc[0]
    print(f"{e['Проект']}, {e['Когда']}: {e['Что случилось']} — "
          f"{e['В этот день']} при обычных {e['Обычно']}. "
          f"Это {e['Насколько сильно']}, и больше аномалий нет: "
          f"обычный разброс в список не попадает.")

## 4. Операторы

Сколько звонков оператор берёт за рабочий день. Смена не задана
заранее, а видна по данным: у кого большая часть звонков в дневные
часы — тот в дневной смене.

In [ ]:
ops = pd.read_sql(text("""
    SELECT operator,
           1.0 * COUNT(*) / COUNT(DISTINCT call_date)          AS calls_per_day,
           AVG(CASE WHEN call_hour BETWEEN :h1 AND :h2
                    THEN 1.0 ELSE 0 END)                       AS day_share,
           ROUND(AVG(talk_sec))                                AS avg_talk_sec
    FROM calls
    WHERE project = :p AND operator IS NOT NULL
    GROUP BY operator
"""), engine, params={"p": MAIN_PROJECT, "h1": DAY_SHIFT[0], "h2": DAY_SHIFT[1]})
ops["Смена"] = np.where(ops["day_share"] >= 0.5, "дневная", "ночная")
ops = ops.sort_values("calls_per_day")

fig, ax = plt.subplots(figsize=(9, 3.8))
day_color = PROJECT_COLORS[MAIN_PROJECT]
colors = [day_color if s == "дневная" else LIGHT_BLUE for s in ops["Смена"]]
# barh — горизонтальные столбцы: имена операторов читаются без наклона
ax.barh(ops["operator"], ops["calls_per_day"], color=colors, height=0.6)
for y, v in enumerate(ops["calls_per_day"]):
    ax.annotate(f"{v:.0f}", (v, y), xytext=(4, 0), textcoords="offset points",
                va="center", color=INK_2)
ax.grid(False)
ax.set_xlabel("звонков за рабочий день, в среднем")
ax.set_title(f"{MAIN_PROJECT}: звонков на оператора за рабочий день")
# Легенда из «пустых» прямоугольников нужного цвета: столбцы разных
# смен нарисованы одним вызовом barh, и своих подписей у них нет
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=day_color),
                   plt.Rectangle((0, 0), 1, 1, color=LIGHT_BLUE)],
          labels=["дневная смена", "ночная смена"], loc="lower right")
plt.show()

display(ops.round(2).sort_values("calls_per_day", ascending=False))

by_shift = ops.groupby("Смена")["calls_per_day"]
day, night = by_shift.median().get("дневная"), by_shift.median().get("ночная")
if day and night:
    print(f"Дневной оператор берёт около {day:.0f} звонков за день, ночной — "
          f"около {night:.0f}. Разница — от потока, а не от старания: "
          f"сравнивать операторов имеет смысл внутри смены.")
spread = by_shift.max() - by_shift.min()
print("Разброс внутри смены, звонков в день:", spread.round(1).to_dict())

## Что видно

- Нагрузка по часам и доля пропущенных — это одна история: пропуски
  растут там, где на линии меньше людей, — утром до выхода смены и
  поздно вечером. Это аргумент сдвинуть смены, а не добавлять людей.
- Аномалия — редкость: из месяца данных в список попадает только
  заложенный в генератор сбой, обычный разброс отсеивается.
- Операторов сравнивают внутри смены: ночной оператор берёт меньше
  не потому, что хуже работает, а потому, что ночью меньше звонков.